In [1]:
import sys
import tempfile
import numpy as np 
import pandas as pd 
from datetime import date, datetime, timedelta
from pathlib import Path 

from mps.core.config import cfg, msg 
from mps.core.types import Bar
from mps.core.calendar import market_open_dt
from mps.core.libs import logger

In [2]:
cfg.sys.timezone, cfg.run.tickers[0]

(zoneinfo.ZoneInfo(key='Asia/Seoul'), '005930')

In [3]:
def make_bar(ticker: str, dt: datetime, price: float = 70_000.0, volume: int = 1000) -> Bar:
    return Bar(
        ticker=ticker,
        timestamp=dt,
        open=price,
        high=price+100,
        low=price-100,
        close=price+50,
        volume=volume,
        is_complete=True
    )

In [4]:
def make_day_bars(ticker: str, work_d: date, n: int = 390) -> list[Bar]:
    """ 하루치 n분봉 생성 핼퍼 """
    work_t = market_open_dt(check_date=work_d)
    return [make_bar(ticker, work_t+timedelta(minutes=idx)) for idx in range(n)]

In [5]:
TICKER: str = cfg.run.tickers[0]

### 1. LocalParquetStore

#### 1-1. 기본 저장·로드 왕복 (round-trip)

In [6]:
from mps.data.io import LocalParquetStore

with tempfile.TemporaryDirectory() as tmp:
    logger.debug(tmp)
    store = LocalParquetStore(base_dir=Path(tmp))
    
    work_date = date(2025, 3, 3)
    work_date_bars = make_day_bars(TICKER, work_date)
    store.save_bars(work_date_bars)
    
    store_path = store._build_store_path(TICKER)
    assert store_path.exists(), "파일이 없음"
    
    start_dt = datetime.combine(work_date, datetime.min.time(), tzinfo=cfg.sys.timezone)
    end_dt = datetime.combine(work_date, datetime.max.time(), tzinfo=cfg.sys.timezone)
    logger.debug(f"{start_dt} ~ {end_dt}")
    load_bars = store.load_bars(TICKER, start_dt, end_dt)
    
    assert len(load_bars) == len(work_date_bars), f"봉 수 불일치: {len(load_bars)} != {len(work_date_bars)}"
    work_bar = load_bars[0]
    assert work_bar.ticker == TICKER
    assert work_bar.open == work_date_bars[0].open 
    assert work_bar.is_complete is True
    
logger.info(f"현재까지 입력된 봉 수: {len(load_bars)}")

KRX 로그인 실패: KRX_ID 또는 KRX_PW 환경 변수가 설정되지 않았습니다.
2026-06-22 16:00:03.635    DEBUG: /tmp/tmpc8nxx1g3
2026-06-22 16:00:03.747    DEBUG: /data/io/_store.py[100]: 저장한 데이터 정보: 데이터 크기=(390, 7), 기간=2025-03-03~2025-03-03
2026-06-22 16:00:03.747    DEBUG: 2025-03-03 00:00:00+09:00 ~ 2025-03-03 23:59:59.999999+09:00
2026-06-22 16:00:03.787    DEBUG: s/data/io/_store.py[60]: 불러온 데이터 정보: 데이터 크기=(390, 7), 기간=2025-03-03~2025-03-03
2026-06-22 16:00:03.790     INFO: 현재까지 입력된 봉 수: 390


#### 1-2. 날짜 구간 필터링

In [7]:
with tempfile.TemporaryDirectory() as tmp:
    store = LocalParquetStore(base_dir=Path(tmp))
    
    date1 = date(2025, 3, 3)
    date2 = date(2025, 3, 4)
    date1_bars = make_day_bars(TICKER, date1)
    date2_bars = make_day_bars(TICKER, date2)
    store.save_bars(date1_bars + date2_bars)
    
    start_dt = datetime.combine(date1, datetime.min.time(), tzinfo=cfg.sys.timezone)
    end_dt = datetime.combine(date1, datetime.max.time(), tzinfo=cfg.sys.timezone)
    load_date1_bars = store.load_bars(TICKER, start_dt, end_dt)
    
    assert len(load_date1_bars) == 390, f"{date1} 봉 수 불일치: {len(load_date1_bars)}"
    assert all(bar.timestamp.date() == date1 for bar in load_date1_bars), f"날짜 필터 오류"

2026-06-22 16:00:03.831    DEBUG: /data/io/_store.py[100]: 저장한 데이터 정보: 데이터 크기=(780, 7), 기간=2025-03-03~2025-03-04
2026-06-22 16:00:03.846    DEBUG: s/data/io/_store.py[60]: 불러온 데이터 정보: 데이터 크기=(390, 7), 기간=2025-03-03~2025-03-03


#### 1-3. 파일이 없을 때 빈 리스트 반한

In [8]:
with tempfile.TemporaryDirectory() as tmp:
    store = LocalParquetStore(base_dir=Path(tmp))
    
    work_date = date(2025, 3, 3)
    start_dt = datetime.combine(work_date, datetime.min.time(), tzinfo=cfg.sys.timezone)
    end_dt = datetime.combine(work_date, datetime.max.time(), tzinfo=cfg.sys.timezone)
    
    load_bars = store.load_bars(TICKER, start_dt, end_dt)
    assert load_bars == [], "파일이 존재하지 않으면 빈 리스트여야 합니다."

2026-06-22 16:00:03.871  WARNING: s/data/io/_store.py[51]: Parquet 파일이 존재하지 않습니다: /tmp/tmpph9pz2ft/005930_minute_bars.parquet


#### 1-4. 중복 timestamp 덮어쓰기

In [9]:
with tempfile.TemporaryDirectory() as tmp:
    store = LocalParquetStore(base_dir=Path(tmp))
    
    work_date = date(2025, 3, 3)
    first_ts = market_open_dt(work_date)
    
    bar_old = make_bar(TICKER, first_ts, price=70_000.0)
    bar_new = make_bar(TICKER, first_ts, price=80_000.0)
    
    store.save_bars([bar_old])
    load_bars = store.load_bars(TICKER, first_ts, first_ts)
    logger.debug(load_bars)
    
    store.save_bars([bar_new])
    load_bars = store.load_bars(TICKER, first_ts, first_ts)
    logger.debug(load_bars)
    
    start_dt = datetime.combine(work_date, datetime.min.time(), tzinfo=cfg.sys.timezone)
    end_dt = datetime.combine(work_date, datetime.max.time(), tzinfo=cfg.sys.timezone)
    load_bars = store.load_bars(TICKER, start_dt, end_dt)
    
    assert len(load_bars) == 1, f"봉 수 불일치: 1 != {len(load_bars)}"
    assert load_bars[0].open == 80000.0, f"최신 데이터 갱신 실패: {load_bars[0].open}"
    

2026-06-22 16:00:03.898    DEBUG: /data/io/_store.py[100]: 저장한 데이터 정보: 데이터 크기=(1, 7), 기간=2025-03-03~2025-03-03
2026-06-22 16:00:03.911    DEBUG: s/data/io/_store.py[60]: 불러온 데이터 정보: 데이터 크기=(1, 7), 기간=2025-03-03~2025-03-03
2026-06-22 16:00:03.911    DEBUG: [Bar(ticker='005930', timestamp=datetime.datetime(2025, 3, 3, 9, 0, tzinfo=<DstTzInfo 'Asia/Seoul' KST+9:00:00 STD>), open=70000.0, high=70100.0, low=69900.0, close=70050.0, volume=1000, is_complete=True)]
2026-06-22 16:00:03.927    DEBUG: /data/io/_store.py[100]: 저장한 데이터 정보: 데이터 크기=(1, 7), 기간=2025-03-03~2025-03-03
2026-06-22 16:00:03.938    DEBUG: s/data/io/_store.py[60]: 불러온 데이터 정보: 데이터 크기=(1, 7), 기간=2025-03-03~2025-03-03
2026-06-22 16:00:03.939    DEBUG: [Bar(ticker='005930', timestamp=datetime.datetime(2025, 3, 3, 9, 0, tzinfo=<DstTzInfo 'Asia/Seoul' KST+9:00:00 STD>), open=80000.0, high=80100.0, low=79900.0, close=80050.0, volume=1000, is_complete=True)]
2026-06-22 16:00:03.950    DEBUG: s/data/io/_store.py[60]: 불러온 데이터 정보: 데이터 크

#### 1-5. 빈 리스트 저장 시 아무것도 안 함.

In [10]:
with tempfile.TemporaryDirectory() as tmp:
    store = LocalParquetStore(base_dir=Path(tmp))
    
    # 빈 bars 저장
    store.save_bars([])
    
    store_path = store._build_store_path(TICKER)
    assert not store_path.exists(), "빈 리스트를 저장할 경우 파일이 생성되지 않음."

#### 1-6. 여러 종목 동시 저장

In [11]:
with tempfile.TemporaryDirectory() as tmp:
    store = LocalParquetStore(base_dir=Path(tmp))
    
    work_date = date(2025, 3, 3)
    tickers: list[str] = ["005930", "000660"]
    t1_bars = make_day_bars(tickers[0], work_date)
    t2_bars = make_day_bars(tickers[1], work_date)
    
    store.save_bars(t1_bars)
    store.save_bars(t2_bars)
    
    start_dt = datetime.combine(work_date, datetime.min.time(), tzinfo=cfg.sys.timezone)
    end_dt = datetime.combine(work_date, datetime.max.time(), tzinfo=cfg.sys.timezone)
    
    t1_load_bars = store.load_bars(tickers[0], start_dt, end_dt)
    t2_load_bars = store.load_bars(tickers[1], start_dt, end_dt)
    
    assert t1_load_bars[0].open == 70000.0
    assert t1_load_bars[0].ticker == tickers[0]
    assert t2_load_bars[0].ticker == tickers[1]

2026-06-22 16:00:03.991    DEBUG: /data/io/_store.py[100]: 저장한 데이터 정보: 데이터 크기=(390, 7), 기간=2025-03-03~2025-03-03
2026-06-22 16:00:04.008    DEBUG: /data/io/_store.py[100]: 저장한 데이터 정보: 데이터 크기=(390, 7), 기간=2025-03-03~2025-03-03
2026-06-22 16:00:04.021    DEBUG: s/data/io/_store.py[60]: 불러온 데이터 정보: 데이터 크기=(390, 7), 기간=2025-03-03~2025-03-03
2026-06-22 16:00:04.036    DEBUG: s/data/io/_store.py[60]: 불러온 데이터 정보: 데이터 크기=(390, 7), 기간=2025-03-03~2025-03-03


### 2. _synthesize_minute_bars or volume_weights 

##### 2-1. 합성 분봉 개수 및 봉 필드 유효성 체크

In [12]:
from mps.data.io._loader import _synthesize_minute_bars, _volume_weights

In [13]:
rng = np.random.default_rng(seed=cfg.sys.seed)
work_date = date(2025, 3, 3)
ohlcv = {
    cfg.key.open    : 70000.0,
    cfg.key.high    : 72000.0,
    cfg.key.low     : 64000.0,
    cfg.key.close   : 71000.0,
    cfg.key.volume  : 1000,
}

bars = _synthesize_minute_bars(TICKER, work_date, ohlcv, rng)
logger.debug(len(bars))
assert len(bars) == cfg.market.minutes_per_day, f'예상 봉 갯 수: {cfg.market.minutes_per_day}, 실제 갯 수: {len(bars)}'

2026-06-22 16:00:04.065    DEBUG: 390


In [14]:
# 모든 봉 유효성 검사
for bar in bars:
    assert bar.ticker == TICKER 
    assert bar.is_complete is True 
    assert bar.high >= bar.low
    assert bar.high >- bar.open and bar.high >= bar.close 
    assert bar.low <= bar.open and bar.low <= bar.close 
    assert bar.volume > 0
    
# 마지막 봉 종가는 일봉 종가
assert bars[-1].close == ohlcv[cfg.key.close]

# 가격이 일봉 고저 범위를 벗어나지 않음
all_prices = [bar.high for bar in bars] + [bar.low for bar in bars]
assert max(all_prices) <= ohlcv[cfg.key.high]
assert min(all_prices) >= ohlcv[cfg.key.low]

#### 2-2. _volume_weights ─ 합산 1.0, KOSPI W형 패턴 확인

In [15]:
rng = np.random.default_rng(seed=cfg.sys.seed)

weights = _volume_weights(bar_count=cfg.market.minutes_per_day, rng=rng)

In [16]:
# 합산 = 1.0
assert abs(weights.sum() - 1.0) < cfg.sys.zero
logger.debug(weights.sum())

# shape
assert weights.shape == (cfg.market.minutes_per_day,)
logger.debug(weights.shape)

2026-06-22 16:00:04.092    DEBUG: 1.0000000000000002
2026-06-22 16:00:04.092    DEBUG: (390,)


### 3. HistoricalDataLoader

In [17]:
import types 
import sys

# pykrx.stock 함수는 네트웍을 타야하는데, 테스트환경에서 네트웍이 안될 수 있으니
# 해당 모듈을 시스템에 가상으로 전달해 후속 코드에서 그 모듈을 사용하도록 stub 코드 작성
def _make_pykrx_stub(dates: list[date], base_price: float = 70000.0) -> types.ModuleType:
    """ pykrx.stock.get_market_ohlcv_by_date() 스텁 주입 """
    mock_data = {
        pd.Timestamp(work_date): {
            '시가': base_price, 
            '고가': base_price * 1.02,
            '저가': base_price * 0.97,
            '종가': base_price * 0.99,
            '거래량': 1_000_000,
        }
        for work_date in dates 
    }
    
    stub_df = pd.DataFrame(mock_data).T
    stub_df.index.name = None 
    
    krx_stub = types.ModuleType('pykrx')
    stock_stub = types.ModuleType('pykrx.stock')
    stock_stub.get_market_ohlcv_by_date = lambda *a, **k: stub_df
    krx_stub.stock = stock_stub
    sys.modules['pykrx'] = krx_stub
    sys.modules['pykrx.stock'] = stock_stub
    return krx_stub

logger.debug('pykrx stub 준비 완료')    

2026-06-22 16:00:04.102    DEBUG: pykrx stub 준비 완료


#### 3-1. 캐시 없을 때 합성 데이터 수집 후 저장

In [18]:
from mps.data.io import LocalParquetStore, HistoricalDataLoader

In [19]:
test_dates = [date(2025, 3, 4), date(2025, 3, 5)]
_make_pykrx_stub(test_dates)

<module 'pykrx'>

In [20]:
logger.point(test_dates)
with tempfile.TemporaryDirectory() as tmp:
    store = LocalParquetStore(base_dir=Path(tmp))
    loader = HistoricalDataLoader(store=store)
    
    source, bars = loader.load(
        TICKER,
        start_date=test_dates[0],
        end_date=test_dates[1],
        force_refresh=False
    )
    
    assert source == cfg.str.pykrx
    expected_bars = len(test_dates) * cfg.market.minutes_per_day
    assert len(bars) == expected_bars
    
    assert store._build_store_path(TICKER).exists()

2026-06-22 16:00:04.128    POINT: [datetime.date(2025, 3, 4), datetime.date(2025, 3, 5)]
2026-06-22 16:00:04.141  WARNING: s/data/io/_store.py[51]: Parquet 파일이 존재하지 않습니다: /tmp/tmpp_yc8lx8/005930_minute_bars.parquet
2026-06-22 16:00:04.340    POINT: /data/io/_loader.py[48]: 데이터 로드 결과: 출처[PYKRX], 크기[780], 기간: 2025-03-04 09:00:00+09:00 ~ 2025-03-05 15:29:00+09:00
2026-06-22 16:00:04.365    DEBUG: /data/io/_store.py[100]: 저장한 데이터 정보: 데이터 크기=(780, 7), 기간=2025-03-04~2025-03-05


#### 3-2. 캐시에 있을때

In [21]:
test_dates = [date(2025, 3, 4)]

with tempfile.TemporaryDirectory() as tmp:
    store = LocalParquetStore(base_dir=Path(tmp))
    loader = HistoricalDataLoader(store=store)
    
    # 현재 데이터가 없으니, 합성함.
    source1, bars1 = loader.load(TICKER, test_dates[0], test_dates[0], force_refresh=False)
    logger.debug(msg.loader.load_result(source1, bars1))
    assert source1 == cfg.str.pykrx
    
    source2, bars2 = loader.load(TICKER, test_dates[0], test_dates[0], force_refresh=False)
    logger.debug(msg.loader.load_result(source2, bars2))
    assert source2 == cfg.str.store
    
    source3, bars3 = loader.load(TICKER, test_dates[0], test_dates[0], force_refresh=True)
    logger.debug(msg.loader.load_result(source3, bars3))
    assert source3 == cfg.str.pykrx
    

2026-06-22 16:00:04.387  WARNING: s/data/io/_store.py[51]: Parquet 파일이 존재하지 않습니다: /tmp/tmppp731yjv/005930_minute_bars.parquet
2026-06-22 16:00:04.544    POINT: /data/io/_loader.py[48]: 데이터 로드 결과: 출처[PYKRX], 크기[390], 기간: 2025-03-04 09:00:00+09:00 ~ 2025-03-04 15:29:00+09:00
2026-06-22 16:00:04.559    DEBUG: /data/io/_store.py[100]: 저장한 데이터 정보: 데이터 크기=(390, 7), 기간=2025-03-04~2025-03-04
2026-06-22 16:00:04.567    DEBUG: _29211/1010043245.py[9]: 데이터 로드 결과: 출처[PYKRX], 크기[390], 기간: 2025-03-04 09:00:00+09:00 ~ 2025-03-04 15:29:00+09:00
2026-06-22 16:00:04.579    DEBUG: s/data/io/_store.py[60]: 불러온 데이터 정보: 데이터 크기=(390, 7), 기간=2025-03-04~2025-03-04
2026-06-22 16:00:04.589    POINT: /data/io/_loader.py[44]: 데이터 로드 결과: 출처[STORE], 크기[390], 기간: 2025-03-04 09:00:00+09:00 ~ 2025-03-04 15:29:00+09:00
2026-06-22 16:00:04.596    DEBUG: 29211/1010043245.py[13]: 데이터 로드 결과: 출처[STORE], 크기[390], 기간: 2025-03-04 09:00:00+09:00 ~ 2025-03-04 15:29:00+09:00
2026-06-22 16:00:04.609    DEBUG: s/data/io/_store.py[60